## Baseline Erf Model

### NB
This cell needs to be run two times because of a package conflict: ood_metrics requires an older numpy
version than the one of colab. Thus, we need to restart the session and run the cell again.

In [ ]:
# @title
!pip install ood_metrics
!pip install lightning > /dev/null
!pip install gitignore_parser > /dev/null
!pip install -U 'jsonargparse[signatures]>=4.27.7' >/dev/null

In [ ]:
# @title
from google.colab import drive
import os
import sys
import json
import yaml
import torch
import importlib
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
from lightning import seed_everything

# 1. Mount Drive and Configure Paths
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

project_root = '/content/drive/MyDrive/FundGitHubProject'
if not os.path.exists('/content/ProjectFolder'):
  # creates shortcut to access the project folder
  !ln -s /content/drive/MyDrive/FundGitHubProject /content/ProjectFolder
eomt_folder = project_root + '/eomt'

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
if eomt_folder not in sys.path:
    sys.path.insert(0, eomt_folder)

from eval.iouEval import iouEval
seed_everything(0, verbose=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")

Mounted at /content/drive
Active Device: cuda


In [ ]:
import torch

erfnet_pretrained = torch.load('trained_models/erfnet_encoder_pretrained.pth.tar')

In [ ]:
import torch
import os
from eomt.datasets.cityscapes_semantic import CityscapesSemantic

# Import the model class (usually named Net or ERFNet in standard implementations)
try:
    from eval.erfnet import Net as ERFNet
except ImportError:
    from eval.erfnet import ERFNet

# Provide the path to the Cityscapes dataset
dataset_path = os.path.join(project_root, 'eomt/data') # adjust this path as needed

# Instantiate the DataModule (it needs self)
cityscapes_dm = CityscapesSemantic(path=dataset_path)

# Lightning DataModules typically require setup before getting dataloaders
cityscapes_dm.setup(stage='validate')
val_dataloader = cityscapes_dm.val_dataloader()

# Remove 'module.' prefix from state dict keys (caused by DataParallel)
# Check if the loaded content is a dictionary and contains 'state_dict'
if isinstance(erfnet_pretrained, dict) and 'state_dict' in erfnet_pretrained:
    state_dict_to_clean = erfnet_pretrained['state_dict']
else:
    state_dict_to_clean = erfnet_pretrained

# Remove both 'module.' and 'features.' prefixes
cleaned_state_dict = {}
for k, v in state_dict_to_clean.items():
    new_k = k.replace('module.', '')
    new_k = new_k.replace('features.', '')
    cleaned_state_dict[new_k] = v

# Initialize the model (Cityscapes typically has 19 or 20 classes)
try:
    model = ERFNet(19)
    # Try to load with strict=False in case there are missing keys (e.g., decoder)
    model.load_state_dict(cleaned_state_dict, strict=False)
except RuntimeError:
    # Fallback to 20 classes if state_dict shapes mismatch
    model = ERFNet(20)
    model.load_state_dict(cleaned_state_dict, strict=False)

# Move model to device and set to evaluation mode
model = model.to(device)
model.eval()

for batch_idx, batch in enumerate(val_dataloader):
    # Lightning dataloaders usually return (images, targets)
    images, targets = batch
    if isinstance(images, tuple):
        images = torch.stack(images)

    # Cast to float to match model weights, scaling to [0, 1]
    images = images.to(device).float() / 255.0

    # Forward pass without calculating gradients
    with torch.no_grad():
        outputs = model(images)

    print(f"Successfully processed batch {batch_idx}! Output shape: {outputs.shape}")
    break


Successfully processed batch 0! Output shape: torch.Size([16, 19, 1024, 2048])


## ERFNet Anomaly Detection Baseline

In [ ]:
print("Running evalAnomaly.py via CLI...")
!cd {project_root}/eval && python evalAnomaly.py \
    --input "Validation_Dataset/RoadAnomaly/images/*.jpg" \
    --loadDir "../trained_models/" \
    --loadWeights "erfnet_pretrained.pth" \
    --loadModel "erfnet.py" | grep -v "images/"

Running evalAnomaly.py via CLI...
Loading model: ../trained_models/erfnet.py
Loading weights: ../trained_models/erfnet_pretrained.pth
Model and weights LOADED successfully
AUPRC score: 15.581983299743523
FPR@TPR95: 73.24763926329555


In [ ]:
print("Running evalAnomaly.py via CLI...")
!cd {project_root}/eval && python evalAnomaly.py \
    --input "Validation_Dataset/RoadAnomaly21/images/*.png" \
    --loadDir "../trained_models/" \
    --loadWeights "erfnet_pretrained.pth" \
    --loadModel "erfnet.py" | grep -v "images/"

Running evalAnomaly.py via CLI...
Loading model: ../trained_models/erfnet.py
Loading weights: ../trained_models/erfnet_pretrained.pth
Model and weights LOADED successfully
AUPRC score: 38.319578121119356
FPR@TPR95: 59.33706864897953


In [ ]:
import os

eval_script = os.path.join(project_root, 'eval', 'evalAnomaly.py')
with open(eval_script, 'r') as f:
    content = f.read()

# Patch the script to replace .webp with .png for ground truth masks
content = content.replace('.replace("images", "labels_masks")', '.replace("images", "labels_masks").replace(".webp", ".png")')
content = content.replace(".replace('images', 'labels_masks')", ".replace('images', 'labels_masks').replace('.webp', '.png')")

with open(eval_script, 'w') as f:
    f.write(content)

print("Running evalAnomaly.py via CLI...")
!cd {project_root}/eval && python evalAnomaly.py \
    --input "Validation_Dataset/RoadObstacle21/images/*.webp" \
    --loadDir "../trained_models/" \
    --loadWeights "erfnet_pretrained.pth" \
    --loadModel "erfnet.py" | grep -v "images/"

Running evalAnomaly.py via CLI...
Loading model: ../trained_models/erfnet.py
Loading weights: ../trained_models/erfnet_pretrained.pth
Model and weights LOADED successfully
AUPRC score: 4.6265677513628045
FPR@TPR95: 48.443441222061765


In [ ]:
import os

# Patch 1: Fix .jpg to .png replacement for ground truth masks
content = content.replace('.replace("images", "labels_masks")', '.replace("images", "labels_masks").replace(".jpg", ".png")')
content = content.replace(".replace('images', 'labels_masks')", ".replace('images', 'labels_masks').replace('.jpg', '.png')")

with open(eval_script, 'w') as f:
    f.write(content)

print("Running evalAnomaly.py via CLI...")
!cd {project_root}/eval && python evalAnomaly.py \
    --input "Validation_Dataset/fs_static/images/*.jpg" \
    --loadDir "../trained_models/" \
    --loadWeights "erfnet_pretrained.pth" \
    --loadModel "erfnet.py" | grep -v "images/"

Running evalAnomaly.py via CLI...
Loading model: ../trained_models/erfnet.py
Loading weights: ../trained_models/erfnet_pretrained.pth
Model and weights LOADED successfully
AUPRC score: 9.498677681348113
FPR@TPR95: 40.300119503862554


## Tests with max entropy

In [10]:
!pip install wget

In [19]:
import os

print("Running evalAnomaly.py via CLI...")
!cd {project_root}/eval && python evalAnomaly.py \
    --input "Validation_Dataset/RoadAnomaly/images/*.jpg" \
    --loadDir "../trained_models/" \
    --loadWeights "erfnet_pretrained.pth" \
    --method "max_entropy" \
    --loadModel "erfnet.py" | grep -v "images/"

Running evalAnomaly.py via CLI...
Loading model: ../trained_models/erfnet.py
Loading weights: ../trained_models/erfnet_pretrained.pth
Scoring method: max_entropy
Results will be saved to: /content/drive/MyDrive/FundGitHubProject/eval/results_anomaly/erfnet_pretrained_max_entropy_20260521_100627
Model and weights LOADED successfully
AUPRC score: 12.668527719053055
FPR@TPR95: 82.74862138731258
Metrics successfully saved to /content/drive/MyDrive/FundGitHubProject/eval/results_anomaly/erfnet_pretrained_max_entropy_20260521_100627/metrics.txt
